# Retraining v2

После мониторинга `production_1` обнаружены:

- data drift;
- prediction drift;
- снижение качества v1 после появления labels.

Обучаем v2 с использованием более свежей истории и затем сравниваем v1 и v2 на следующем временном периоде `production_2`.

In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from lightgbm import LGBMClassifier

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)

## Загружаем старые splits:

In [2]:
train = pd.read_parquet(
    "../data/processed/train.parquet"
)

validation = pd.read_parquet(
    "../data/processed/validation.parquet"
)

test = pd.read_parquet(
    "../data/processed/test.parquet"
)

for df in [train, validation, test]:
    df["booking_date"] = pd.to_datetime(
        df["booking_date"]
    )

## Восстанавливаем production_1 и production_2:

In [3]:
production_1 = test[
    (test["booking_date"] >= "2017-02-01")
    & (test["booking_date"] < "2017-05-01")
].copy()

production_2 = test[
    test["booking_date"] >= "2017-05-01"
].copy()

## Информация, которая к моменту retraining уже известна:

In [5]:
retraining_history = pd.concat(
    [
        train,
        validation,
        production_1,
    ],
    ignore_index=True,
).sort_values("booking_date")

## Новый temporal split для v2

Последний месяц оставим для validation

In [6]:
v2_train = retraining_history[
    retraining_history["booking_date"]
    < "2017-04-01"
].copy()

v2_val = retraining_history[
    retraining_history["booking_date"]
    >= "2017-04-01"
].copy()

## Подготовка X и Y

In [7]:
target = "is_canceled"

drop_cols = [
    target,
    "booking_date",
    "arrival_date",
]

X_train_v2 = v2_train.drop(
    columns=drop_cols
)

y_train_v2 = v2_train[target]

X_val_v2 = v2_val.drop(
    columns=drop_cols
)

y_val_v2 = v2_val[target]

In [12]:
categorical_cols = (
    X_train_v2
    .select_dtypes(
        include=[
            "object",
            "string",
            "category",
        ]
    )
    .columns
    .tolist()
)

numeric_cols = (
    X_train_v2
    .select_dtypes(include=["number"])
    .columns
    .tolist()
)


## Новый preprocessor

Training data изменилась, поэтому старый использовать нельзя

In [13]:
numeric_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median"),
    ),
    (
        "scaler",
        StandardScaler(),
    ),
])

categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="constant",
            fill_value="Missing",
        ),
    ),
    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore",
        ),
    ),
])

preprocessor_v2 = ColumnTransformer([
    (
        "num",
        numeric_pipeline,
        numeric_cols,
    ),
    (
        "cat",
        categorical_pipeline,
        categorical_cols,
    ),
])

In [14]:
X_train_v2_enc = (
    preprocessor_v2.fit_transform(
        X_train_v2
    )
)

X_val_v2_enc = (
    preprocessor_v2.transform(
        X_val_v2
    )
)

print(X_train_v2_enc.shape)
print(X_val_v2_enc.shape)

(110075, 900)
(2736, 900)


## Обучаем lightGBM v2

Возьмем старые параметры

In [15]:
model_v2 = LGBMClassifier(
    n_estimators=399,
    learning_rate=0.02,
    num_leaves=95,
    max_depth=-1,
    min_child_samples=20,
    subsample=0.70,
    subsample_freq=1,
    colsample_bytree=0.70,
    reg_alpha=0.0,
    reg_lambda=1.0,
    objective="binary",
    random_state=42,
    n_jobs=-1,
    verbosity=-1,
)

model_v2.fit(
    X_train_v2_enc,
    y_train_v2,
)

,num_leaves,95
,learning_rate,0.02
,n_estimators,399
,objective,'binary'
,subsample,0.7
,subsample_freq,1
,colsample_bytree,0.7
,reg_lambda,1.0
,random_state,42
,n_jobs,-1
,verbosity,-1


In [17]:
val_v2_proba = np.asarray(
    model_v2.predict_proba(X_val_v2_enc)
)[:, 1]

ranking_metrics = {
    "roc_auc": roc_auc_score(
        y_val_v2,
        val_v2_proba,
    ),
    "pr_auc": average_precision_score(
        y_val_v2,
        val_v2_proba,
    ),
}

pd.Series(ranking_metrics).round(4)

roc_auc    0.8736
pr_auc     0.7438
dtype: float64

Пока модель не выглядит сильнее, но это пока не полная проверка, полная проверка будет на production_2

## Выбор нового порога

Сейчас сначала выбираем новый threshold v2 по тому же правилу, что и раньше: Recall >= 0.90, а среди таких порогов максимизируем precision.

In [19]:
from sklearn.metrics import precision_recall_curve

In [20]:
precision, recall, thresholds = precision_recall_curve(
    y_val_v2,
    val_v2_proba,
)

threshold_table = pd.DataFrame({
    "threshold": thresholds,
    "precision": precision[:-1],
    "recall": recall[:-1],
})

candidates = threshold_table[
    threshold_table["recall"] >= 0.90
].copy()

best_row = (
    candidates
    .sort_values(
        ["precision", "threshold"],
        ascending=[False, False],
    )
    .iloc[0]
)

v2_threshold = float(
    best_row["threshold"]
)

best_row

threshold    0.175809
precision    0.551006
recall       0.900235
Name: 1251, dtype: float64

In [21]:
v2_threshold

0.17580904504576347

### Посмотрим все метрики на новом пороге:


In [22]:
val_v2_pred = (
    val_v2_proba >= v2_threshold
).astype(int)

v2_validation_metrics = pd.Series({
    "precision": precision_score(
        y_val_v2,
        val_v2_pred,
    ),
    "recall": recall_score(
        y_val_v2,
        val_v2_pred,
    ),
    "f1": f1_score(
        y_val_v2,
        val_v2_pred,
    ),
    "roc_auc": roc_auc_score(
        y_val_v2,
        val_v2_proba,
    ),
    "pr_auc": average_precision_score(
        y_val_v2,
        val_v2_proba,
    ),
})

v2_validation_metrics.round(4)

precision    0.5510
recall       0.9002
f1           0.6836
roc_auc      0.8736
pr_auc       0.7438
dtype: float64

## V1 vs V2

### Прячем labels production_2

In [23]:
production_2_labels = production_2[
    "is_canceled"
].copy()

production_2_live = production_2.drop(
    columns="is_canceled"
).copy()

assert "is_canceled" not in production_2_live.columns

### Прогноз v1

In [24]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.inference import CancellationPredictor

In [25]:
predictor_v1 = CancellationPredictor()

v1_predictions = predictor_v1.predict_batch(
    production_2_live
)

v1_predictions.head()

,prediction,cancellation_probability,model_version
409,0,0.161372,v1
414,0,0.168633,v1
438,0,0.124266,v1
439,0,0.124266,v1
440,0,0.124266,v1


### Прогноз v2

In [27]:
X_prod2_v2 = production_2.drop(
    columns=drop_cols
).copy()

In [28]:
for col in categorical_cols:
    X_prod2_v2[col] = (
        X_prod2_v2[col]
        .astype("object")
        .where(
            X_prod2_v2[col].notna(),
            np.nan,
        )
    )

In [29]:
X_prod2_v2_enc = preprocessor_v2.transform(
    X_prod2_v2
)

In [30]:
v2_proba = np.asarray(
    model_v2.predict_proba(
        X_prod2_v2_enc
    )
)[:, 1]

v2_pred = (
    v2_proba >= v2_threshold
).astype(int)

### Теперь с реальными labels

In [31]:
y_prod2 = production_2_labels

assert y_prod2.index.equals(
    v1_predictions.index
)

V1:

In [32]:
v1_pred = v1_predictions[
    "prediction"
].to_numpy()

v1_proba = v1_predictions[
    "cancellation_probability"
].to_numpy()

### Считаем метрики


In [33]:
def calculate_metrics(
    y_true,
    y_pred,
    y_proba,
):
    return {
        "precision": precision_score(
            y_true,
            y_pred,
            zero_division=0,
        ),
        "recall": recall_score(
            y_true,
            y_pred,
            zero_division=0,
        ),
        "f1": f1_score(
            y_true,
            y_pred,
            zero_division=0,
        ),
        "roc_auc": roc_auc_score(
            y_true,
            y_proba,
        ),
        "pr_auc": average_precision_score(
            y_true,
            y_proba,
        ),
    }

In [34]:
v1_prod2_metrics = calculate_metrics(
    y_prod2,
    v1_pred,
    v1_proba,
)

v2_prod2_metrics = calculate_metrics(
    y_prod2,
    v2_pred,
    v2_proba,
)

In [35]:
model_comparison = pd.DataFrame({
    "v1": v1_prod2_metrics,
    "v2": v2_prod2_metrics,
})

model_comparison["v2_minus_v1"] = (
    model_comparison["v2"]
    - model_comparison["v1"]
)

model_comparison.round(4)

,v1,v2,v2_minus_v1
precision,0.5902,0.5107,-0.0795
recall,0.7244,0.8797,0.1553
f1,0.6504,0.6462,-0.0042
roc_auc,0.8567,0.8645,0.0078
pr_auc,0.6567,0.6910,0.0343


### Поведение моделей:

In [36]:
prediction_rate_comparison = pd.Series({
    "actual_cancellation_rate":
        y_prod2.mean(),

    "v1_predicted_rate":
        v1_pred.mean(),

    "v2_predicted_rate":
        v2_pred.mean(),

    "v1_mean_probability":
        v1_proba.mean(),

    "v2_mean_probability":
        v2_proba.mean(),
})

prediction_rate_comparison.round(4)

actual_cancellation_rate    0.2692
v1_predicted_rate           0.3304
v2_predicted_rate           0.4637
v1_mean_probability         0.2809
v2_mean_probability         0.2268
dtype: float64

## Вывод

Retraining на более свежих данных улучшил ранжирование модели на следующем временном периоде: ROC-AUC вырос с 0.857 до 0.865, а PR-AUC — с 0.657 до 0.691.

Особенно заметно вырос recall: с 0.724 у v1 до 0.880 у v2. При этом precision снизился с 0.590 до 0.511, а F1 практически не изменился.

Порог v2, подобранный на апрельском validation, оказался слишком агрессивным на следующем периоде: модель предсказывает отмену для 46.4% бронирований при фактической доле отмен 26.9%.

Таким образом, обновление training data улучшило способность модели ранжировать риск, но выбранный порог плохо перенёсся на следующий временной период. В текущем виде v2 не стоит выкатывать.

Но и порог я бы не менял, нам все-равно придется делать компромисс между precision и recall, улучшить обе метрики можно выбором новой модели или признаков.